# modeling — `data/features` の把握

- **入力**: `data/features/**/*.parquet`（馬シャードなどファイル数が多いものは論理グループに集約）
- **出力**: `data/features_info/by_dataset/<dataset_key>.json` と `_index.json`
- 各 JSON にカラム型・欠損・（数値は分位数・文字列は上位頻度）を記録。分布は **代表ファイルを最大 20 万行まで読み** 算出（メモリに注意）。


In [6]:
from __future__ import annotations

import json
import math
import sys
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    tqdm = None

# プロジェクトルート（ノートの cwd がどこでも追えるように）
_root = Path.cwd().resolve()
for _ in range(16):
    if (_root / "requirements.txt").is_file() and (_root / ".env.example").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("keiba-vpn ルートが見つかりません")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

REPO_ROOT = _root
FEATURES_ROOT = REPO_ROOT / "data" / "features"
INFO_ROOT = REPO_ROOT / "data" / "features_info"
BY_DATASET = INFO_ROOT / "by_dataset"

print("REPO_ROOT      =", REPO_ROOT)
print("FEATURES_ROOT  =", FEATURES_ROOT)
print("INFO_ROOT      =", INFO_ROOT)


REPO_ROOT      = /home/hirokiakataoka/project/myproject/keiba-vpn
FEATURES_ROOT  = /home/hirokiakataoka/project/myproject/keiba-vpn/data/features
INFO_ROOT      = /home/hirokiakataoka/project/myproject/keiba-vpn/data/features_info


In [7]:
def dataset_group_key(rel: Path) -> str:
    """features/ からの相対パスを論理データセット名に正規化する。"""
    parts = rel.parts
    stem = rel.stem
    if parts and parts[0] == "horse" and len(parts) >= 4:
        return f"horse/{parts[1]}"
    if len(parts) >= 3:
        y = parts[-2]
        if y.isdigit() and len(y) == 4:
            prefix = parts[:-2]
            return "/".join(prefix + (stem,))
    if len(parts) == 2:
        if stem.isdigit() and len(stem) == 4:
            return parts[0]
    return rel.with_suffix("").as_posix()


def json_sanitize(obj):
    if isinstance(obj, dict):
        return {str(k): json_sanitize(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_sanitize(v) for v in obj]
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return float(obj)
    if isinstance(obj, (np.floating, np.integer)):
        v = obj.item()
        return json_sanitize(v)
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if obj is None or isinstance(obj, (str, int, bool)):
        return obj
    return str(obj)


def safe_dataset_filename(key: str) -> str:
    return (
        key.replace("/", "__")
        .replace(" ", "_")
        .replace("\\", "_")
    )


def row_count_fast(path: Path) -> int:
    return int(pq.ParquetFile(path).metadata.num_rows)


def read_sample_dataframe(path: Path, max_rows: int = 200_000) -> pd.DataFrame:
    import pyarrow as pa

    pf = pq.ParquetFile(path)
    n = pf.metadata.num_rows
    if n == 0:
        return pd.DataFrame()
    # 行数が max_rows 以下なら全体読込でよい
    if n <= max_rows:
        return pq.read_table(path).to_pandas()
    # 巨大ファイルは pq.read_table が全行読むため固まる → 行グループ単位で先頭 max_rows 行だけ
    chunks: list = []
    collected = 0
    for rg_idx in range(pf.num_row_groups):
        tbl = pf.read_row_group(rg_idx)
        remain = max_rows - collected
        if tbl.num_rows > remain:
            tbl = tbl.slice(0, remain)
            chunks.append(tbl)
            break
        chunks.append(tbl)
        collected += tbl.num_rows
        if collected >= max_rows:
            break
    if not chunks:
        return pd.DataFrame()
    table = pa.concat_tables(chunks)
    return table.to_pandas()


def series_profile(s: pd.Series, max_cats: int = 40) -> dict:
    n = len(s)
    nulls = int(s.isna().sum())
    out = {
        "dtype": str(s.dtype),
        "null_count": nulls,
        "null_pct": round(nulls / n * 100, 6) if n else 0.0,
    }
    if n == 0:
        return out

    if pd.api.types.is_bool_dtype(s):
        vc = s.value_counts(dropna=False).head(50)
        out["value_counts_top"] = {str(k): int(v) for k, v in vc.items()}
        return out

    if pd.api.types.is_numeric_dtype(s):
        sn = pd.to_numeric(s, errors="coerce")
        desc = sn.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
        dist = {}
        for k, v in desc.items():
            dist[k] = float(v) if pd.notna(v) else None
        nu = sn.nunique(dropna=True)
        out["n_unique"] = int(nu)
        out["distribution"] = dist
        return out

    nu = s.nunique(dropna=False)
    out["n_unique"] = int(nu)
    vc = s.astype("string").fillna("<NA>").value_counts(dropna=False).head(min(max_cats, 50))
    out["value_counts_top"] = {str(k): int(v) for k, v in vc.items()}
    return out


def profile_dataframe(df: pd.DataFrame) -> dict:
    return {col: series_profile(df[col]) for col in df.columns}


In [8]:
assert FEATURES_ROOT.is_dir(), f"見つかりません: {FEATURES_ROOT}"

parquet_paths: list[Path] = sorted(FEATURES_ROOT.rglob("*.parquet"))
groups: dict[str, list[Path]] = defaultdict(list)
for p in parquet_paths:
    rel = p.relative_to(FEATURES_ROOT)
    groups[dataset_group_key(rel)].append(p)

print(f"parquet ファイル数: {len(parquet_paths)}")
print(f"論理データセット数: {len(groups)}")


parquet ファイル数: 72033
論理データセット数: 65


In [9]:
BY_DATASET.mkdir(parents=True, exist_ok=True)

generated_at = datetime.now(timezone.utc).isoformat()
index_entries: list[dict] = []

group_items = sorted(groups.items(), key=lambda x: x[0])
iterator = tqdm(group_items, desc="datasets") if tqdm else group_items

for key, paths in iterator:
    paths = sorted(paths)
    total_rows = 0
    for p in paths:
        total_rows += row_count_fast(p)
    sample_path = paths[0]
    schema = pq.read_schema(sample_path)
    columns_schema = [
        {"name": field.name, "type": str(field.type), "nullable": field.nullable}
        for field in schema
    ]

    df_sample = read_sample_dataframe(sample_path)
    profile = profile_dataframe(df_sample)

    rel_paths = [str(p.relative_to(REPO_ROOT)) for p in paths]

    payload = {
        "dataset_key": key,
        "generated_at": generated_at,
        "sample_parquet": str(sample_path.relative_to(REPO_ROOT)),
        "parquet_files": len(paths),
        "parquet_paths_relative": rel_paths[:500],
        "paths_truncated": len(rel_paths) > 500,
        "total_rows_meta_sum": total_rows,
        "sample_rows_used_for_profile": int(len(df_sample)),
        "pyarrow_schema": columns_schema,
        "pandas_profiles": json_sanitize(profile),
    }

    out_name = safe_dataset_filename(key) + ".json"
    out_path = BY_DATASET / out_name
    out_path.write_text(
        json.dumps(json_sanitize(payload), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    index_entries.append(
        {
            "dataset_key": key,
            "n_parquet_files": len(paths),
            "total_rows_meta_sum": total_rows,
            "n_columns": len(columns_schema),
            "json_relative": str(out_path.relative_to(REPO_ROOT)),
        }
    )

index_payload = {
    "generated_at": generated_at,
    "features_root": str(FEATURES_ROOT.relative_to(REPO_ROOT)),
    "info_root": str(INFO_ROOT.relative_to(REPO_ROOT)),
    "n_datasets": len(index_entries),
    "datasets": sorted(index_entries, key=lambda d: d["dataset_key"]),
}
index_path = INFO_ROOT / "_index.json"
index_path.write_text(
    json.dumps(json_sanitize(index_payload), ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("書き出し完了:", index_path)


datasets: 100%|██████████| 65/65 [00:19<00:00,  3.33it/s]

書き出し完了: /home/hirokiakataoka/project/myproject/keiba-vpn/data/features_info/_index.json


### 一覧（`_index.json` ベース）


In [10]:
from IPython.display import display

idx = json.loads((INFO_ROOT / "_index.json").read_text(encoding="utf-8"))
df_idx = pd.DataFrame(idx["datasets"])
if not df_idx.empty:
    df_idx = df_idx.sort_values("dataset_key").reset_index(drop=True)
display(df_idx)


,dataset_key,n_parquet_files,total_rows_meta_sum,n_columns,json_relative
0,base_tbl/shutuba,6,286060,4,data/features_info/by_dataset/base_tbl__shutub...
1,horse/ped_tbl,35891,1112621,26,data/features_info/by_dataset/horse__ped_tbl.json
2,horse/result_tbl,35891,286060,36,data/features_info/by_dataset/horse__result_tb...
3,race_horse/current_weight_advantage_points,1,47497,3,data/features_info/by_dataset/race_horse__curr...
4,race_horse/field_relative_pre_rating,1,47497,3,data/features_info/by_dataset/race_horse__fiel...
...,...,...,...,...,...
60,snapshots/race_performance_2020_2025,1,283714,40,data/features_info/by_dataset/snapshots__race_...
61,snapshots/race_performance_2025_2025,1,47497,40,data/features_info/by_dataset/snapshots__race_...
62,target/rank_tbl/rank,6,286060,3,data/features_info/by_dataset/target__rank_tbl...
63,trainer_tbl/lookup_trainer,1,349,85,data/features_info/by_dataset/trainer_tbl__loo...


### インタラクティブ閲覧（「戻る」「次へ」）

各論理データセット（`groups` のキー）について、代表 Parquet の先頭から最大 **PREVIEW_MAX_ROWS** 行を読み込み、**カラム一覧（型・ランダムサンプル1件）**・**先頭10行**・**`describe(include='all')`** を表示します。**戻る** / **次へ** で前後のデータセットへ移動します（端で循環）。


In [ ]:
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

PREVIEW_MAX_ROWS = 100_000  # describe 用。馬シャード等でメモリを抑える

_dataset_keys = sorted(groups.keys())
_explorer_idx = {"i": 0}

def _load_preview_dataframe(paths: list[Path]) -> tuple[pd.DataFrame, Path]:
    sample_path = sorted(paths)[0]
    df = read_sample_dataframe(sample_path, max_rows=PREVIEW_MAX_ROWS)
    return df, sample_path

def _column_samples_table(df: pd.DataFrame) -> pd.DataFrame:
    """各カラムのデータ型と、そのカラムから無作為に1件取り出した値（文字列表現）。"""
    rng = np.random.default_rng()
    rows: list[dict] = []
    for col in df.columns:
        ser = df[col]
        dt = str(ser.dtype)
        if len(ser) == 0:
            sample_repr = None
        else:
            idx = int(rng.integers(0, len(ser)))
            val = ser.iloc[idx]
            try:
                txt = repr(val)
                if len(txt) > 300:
                    txt = txt[:300] + "…"
            except Exception:
                txt = str(val)[:300]
            sample_repr = txt
        rows.append({"column": col, "dtype": dt, "random_sample": sample_repr})
    return pd.DataFrame(rows)

def _display_df_scrollable(df: pd.DataFrame, *, max_height: str | None = None) -> None:
    """横幅が広い表を横スクロールで見られるようにする。セル折り返しを抑えて列が途切れにくくする。"""
    frag = df.to_html(notebook=True, border=1, max_rows=None, max_cols=None)
    extra = f"max-height:{max_height};overflow-y:auto;" if max_height else ""
    outer = f"overflow-x:auto;width:100%;{extra}"
    css = (
        "<style scoped>"
        ".df-xscroll table.dataframe td,.df-xscroll table.dataframe th{white-space:nowrap;}"
        "</style>"
    )
    display(HTML(f'<div class="df-xscroll" style="{outer}">{css}{frag}</div>'))

_w_out = widgets.Output(
    layout={"border": "1px solid #ddd", "padding": "10px", "min_height": "120px"}
)
_status = widgets.HTML(value="")

def _render_explorer() -> None:
    key = _dataset_keys[_explorer_idx["i"]]
    paths = sorted(groups[key])
    print(
        f"[explorer] Parquet 読み込み中… [{_explorer_idx['i'] + 1}/{len(_dataset_keys)}] {key}",
        flush=True,
    )
    df, sample_path = _load_preview_dataframe(paths)
    _status.value = (
        "<div><b>[{} / {}]</b> <code>{}</code><br/>"
        "代表ファイル: <code>{}</code> · Parquet {} ファイル · "
        "<code>describe</code> 用に読み込んだ行数 {}</div>"
    ).format(
        _explorer_idx["i"] + 1,
        len(_dataset_keys),
        key,
        sample_path.relative_to(REPO_ROOT),
        len(paths),
        len(df),
    )
    with _w_out:
        clear_output(wait=True)
        print("--- カラム一覧（dtype・ランダム1件） ---")
        _display_df_scrollable(_column_samples_table(df))
        print("\n--- 先頭 10 行 ---")
        _display_df_scrollable(df.head(10))
        print("\n--- df.describe(include='all') ---")
        try:
            desc = df.describe(include="all", datetime_is_numeric=True)
        except Exception as exc:  # pragma: no cover
            print(f"describe(include='all') が失敗したため数値のみ: {exc}")
            desc = df.describe()
        _display_df_scrollable(desc)

_btn_prev = widgets.Button(
    description="戻る",
    tooltip="前の論理データセットへ（先頭で末尾へ）",
)
_btn_next = widgets.Button(
    description="次へ",
    button_style="primary",
    tooltip="次の論理データセットへ（最後で先頭へ）",
)

def _on_prev(_btn) -> None:
    _explorer_idx["i"] = (_explorer_idx["i"] - 1) % len(_dataset_keys)
    _render_explorer()

def _on_next(_btn) -> None:
    _explorer_idx["i"] = (_explorer_idx["i"] + 1) % len(_dataset_keys)
    _render_explorer()

_btn_prev.on_click(_on_prev)
_btn_next.on_click(_on_next)

_box = widgets.VBox([widgets.HBox([_btn_prev, _btn_next, _status]), _w_out])
display(_box)
_render_explorer()
